# Experiment Analysis

**Philosophy:** The experiment ran — now what? Each section covers one step in the analysis pipeline, with explicit attention to the interview pitfalls that trip people up most often.

---

## Decision Table

| If you need to... | Go to |
| :--- | :--- |
| Know what order to run the analysis steps | §1 — Analysis Pipeline |
| Compute ATE, p-value, confidence interval | §2 — Statistical Testing |
| Reduce variance to get results faster | §3 — CUPED |
| Test multiple metrics or segments | §4 — Multiple Testing |
| Diagnose why results look wrong | §5 — Pitfalls & Diagnostics |
| Explain results to a non-technical stakeholder | §6 — Communicating Results |

---
## §1 — The Analysis Pipeline

Always follow this order. Steps 1–3 are validity checks — if any fail, stop before looking at the primary metric.

```
Step 1  SRM check                Did the groups receive the expected traffic split?
         └── If fails → stop. Results are invalid.

Step 2  Covariate balance        Are pre-experiment characteristics equal across groups?
         └── If fails → stop. Randomization is broken.

Step 3  Data quality             Outliers? Missing data? Logging gaps?
         └── Winsorize extreme values before computing means.

Step 4  Primary metric           Compute ATE, 95% CI, p-value.
         └── Report effect size AND practical significance, not just p-value.

Step 5  Guardrail metrics        Did any guardrail degrade?
         └── If yes → do not ship, even if primary metric is positive.

Step 6  Segment analysis         Does the effect differ by platform, country, user tenure?
         └── Apply multiple testing correction. Treat as exploratory.

Step 7  Recommend                Ship / no-ship / iterate — with justification.
```

---
## §2 — Statistical Testing

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

np.random.seed(42)

# ── Test A: Continuous metric — two-sample t-test ─────────────────────────────
# Use when: metric is continuous (session length, revenue per user)
n = 5_000
ctrl = np.random.normal(loc=12.0, scale=5.0, size=n)   # control: mean=12
trt  = np.random.normal(loc=12.6, scale=5.0, size=n)   # treatment: mean=12.6

ate   = trt.mean() - ctrl.mean()
se    = np.sqrt(ctrl.std()**2/n + trt.std()**2/n)
ci_lo = ate - 1.96 * se
ci_hi = ate + 1.96 * se
t_stat, p_val = stats.ttest_ind(trt, ctrl)
cohens_d = ate / np.sqrt((ctrl.std()**2 + trt.std()**2) / 2)

print("── Continuous metric (t-test) ──")
print(f"Control mean:    {ctrl.mean():.3f}")
print(f"Treatment mean:  {trt.mean():.3f}")
print(f"ATE:             {ate:+.3f}  ({ate/ctrl.mean():.2%} relative lift)")
print(f"95% CI:          [{ci_lo:.3f}, {ci_hi:.3f}]")
print(f"p-value:         {p_val:.4f}")
print(f"Cohen's d:       {cohens_d:.3f}  {'(small)' if abs(cohens_d)<0.3 else '(medium)' if abs(cohens_d)<0.5 else '(large)'}")
print()

# ── Test B: Proportion metric — z-test for proportions ───────────────────────
# Use when: metric is a rate (conversion rate, click-through rate)
n_c, conv_c = 10_000, 980    # control: 9.8% conversion
n_t, conv_t = 10_000, 1_050  # treatment: 10.5% conversion

p_c = conv_c / n_c
p_t = conv_t / n_t
p_pool = (conv_c + conv_t) / (n_c + n_t)
se_prop = np.sqrt(p_pool * (1-p_pool) * (1/n_c + 1/n_t))
z_stat  = (p_t - p_c) / se_prop
p_val_prop = 2 * (1 - stats.norm.cdf(abs(z_stat)))
ci_prop = (p_t - p_c) + np.array([-1, 1]) * 1.96 * np.sqrt(p_c*(1-p_c)/n_c + p_t*(1-p_t)/n_t)

print("── Proportion metric (z-test) ──")
print(f"Control rate:    {p_c:.3%}")
print(f"Treatment rate:  {p_t:.3%}")
print(f"Absolute lift:   {p_t-p_c:+.4f}  ({(p_t-p_c)/p_c:.2%} relative)")
print(f"95% CI:          [{ci_prop[0]:.4f}, {ci_prop[1]:.4f}]")
print(f"z-statistic:     {z_stat:.3f}")
print(f"p-value:         {p_val_prop:.4f}")
print()

# ── Statistical vs Practical Significance ────────────────────────────────────
# Statistical significance: p < alpha — effect unlikely due to chance
# Practical significance:   effect size is large enough to matter for the business
# With large n, even tiny effects are statistically significant — always check both

# Cohen's d interpretation:
# 0.2 = small, 0.5 = medium, 0.8 = large

# ── Non-parametric test — Mann-Whitney U ─────────────────────────────────────
# Use when: metric is heavily skewed, small sample, or ordinal
# Tests whether one distribution is stochastically greater than the other
u_stat, p_mw = stats.mannwhitneyu(trt, ctrl, alternative='two-sided')
print(f"── Non-parametric (Mann-Whitney U) ──")
print(f"U-statistic: {u_stat:.0f},  p-value: {p_mw:.4f}")

# ── Visualization ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor('#0b0e14')
for ax in axes:
    ax.set_facecolor('#13171f')
    ax.tick_params(colors='#6b7a99')
    ax.spines[:].set_color('#1e2433')

# Distribution overlap
axes[0].hist(ctrl, bins=60, alpha=0.6, color='#f0c674', label=f'Control μ={ctrl.mean():.2f}')
axes[0].hist(trt,  bins=60, alpha=0.6, color='#4fc4cf', label=f'Treatment μ={trt.mean():.2f}')
axes[0].axvline(ctrl.mean(), color='#f0c674', lw=2, ls='--')
axes[0].axvline(trt.mean(),  color='#4fc4cf', lw=2, ls='--')
axes[0].set_title('Outcome Distributions', color='#d4dbe8')
axes[0].legend(labelcolor='#d4dbe8', facecolor='#1e2433', edgecolor='#1e2433')
axes[0].grid(True, alpha=0.15, color='#1e2433')

# Confidence interval plot
axes[1].errorbar([1], [ate], yerr=[[ate-ci_lo], [ci_hi-ate]],
                 fmt='o', color='#4fc4cf', capsize=12, capthick=2, markersize=10, lw=2)
axes[1].axhline(0, color='#f76f6f', lw=1.5, ls='--', label='Null (no effect)')
axes[1].set_xlim(0, 2)
axes[1].set_xticks([1]); axes[1].set_xticklabels(['ATE'], color='#d4dbe8')
axes[1].set_title(f'ATE = {ate:+.3f}  (p = {p_val:.4f})', color='#d4dbe8')
axes[1].legend(labelcolor='#d4dbe8', facecolor='#1e2433', edgecolor='#1e2433')
axes[1].grid(True, alpha=0.15, color='#1e2433')

plt.tight_layout()
plt.show()

**Which test to use:**

| Metric type | Test | When to switch |
| :--- | :--- | :--- |
| Continuous, approx. normal | Two-sample t-test | — |
| Binary / proportion | Z-test for proportions | — |
| Heavily skewed, small n | Mann-Whitney U | Revenue metrics, session counts |
| Count data (sessions, events) | Bootstrap | When distribution is unknown |

**Common mistakes:**
- Reporting only p-value without effect size — a p=0.001 with Cohen's d=0.02 is statistically significant but practically useless
- Using a one-sided test to get a lower p-value — only valid if you pre-specified the direction; using it post-hoc is p-hacking
- Ignoring outliers before computing means — one user with $10M revenue can swing the ATE; always Winsorize or check distribution first

---
## §3 — Variance Reduction (CUPED)

CUPED (Controlled-experiment Using Pre-Experiment Data) reduces outcome variance by removing the component explainable by a pre-experiment covariate. Same statistical power with a smaller sample — or faster experiments at the same power.

In [ ]:
# ── CUPED: the idea ───────────────────────────────────────────────────────────
# Y_adjusted = Y - theta * (X - E[X])
# X  = pre-experiment covariate (e.g. revenue in the week before the experiment)
# theta = Cov(Y, X) / Var(X)  — OLS coefficient from regressing Y on X
# Result: Y_adjusted has the same mean as Y but lower variance
# Variance reduction factor ≈ 1 - Corr(Y, X)²

np.random.seed(42)
n = 5_000

# Simulate: pre-experiment revenue (X) correlates with outcome (Y)
X_pre = np.random.normal(10, 3, n)                        # pre-experiment revenue
noise = np.random.normal(0, 2, n)
Y_ctrl = 0.8 * X_pre + noise                             # control outcome
Y_trt  = 0.8 * X_pre + noise + 0.5                      # treatment adds +0.5

# CUPED adjustment
def cuped_adjust(Y, X):
    theta = np.cov(Y, X)[0, 1] / np.var(X)              # OLS slope
    Y_adj = Y - theta * (X - X.mean())
    return Y_adj, theta

Y_ctrl_adj, theta_c = cuped_adjust(Y_ctrl, X_pre)
Y_trt_adj,  theta_t = cuped_adjust(Y_trt,  X_pre)

# Compare variance before and after
corr = np.corrcoef(np.concatenate([Y_ctrl, Y_trt]),
                   np.concatenate([X_pre,  X_pre]))[0,1]
var_reduction = 1 - corr**2

print(f"Correlation(Y, X_pre):          {corr:.3f}")
print(f"Variance reduction factor:      {var_reduction:.3f}  ({var_reduction:.1%} of variance removed)")
print()
print(f"Std(Y_ctrl) before CUPED:       {Y_ctrl.std():.3f}")
print(f"Std(Y_ctrl) after  CUPED:       {Y_ctrl_adj.std():.3f}")
print()

# Effect estimate should be the same — CUPED only reduces variance
ate_raw  = Y_trt.mean()     - Y_ctrl.mean()
ate_cuped = Y_trt_adj.mean() - Y_ctrl_adj.mean()
print(f"ATE (raw):         {ate_raw:.4f}")
print(f"ATE (CUPED):       {ate_cuped:.4f}   ← same estimate, lower variance")
print()

# Compare p-values
_, p_raw   = stats.ttest_ind(Y_trt,     Y_ctrl)
_, p_cuped = stats.ttest_ind(Y_trt_adj, Y_ctrl_adj)
print(f"p-value (raw):     {p_raw:.4f}")
print(f"p-value (CUPED):   {p_cuped:.4f}   ← tighter confidence interval")

**Key points:**

- CUPED does not change the ATE estimate — it only reduces variance
- The stronger the correlation between X and Y, the more variance is removed
- `X` must be from **before** the experiment starts — using post-experiment covariates introduces leakage
- Variance reduction factor = `1 - Corr(Y, X)²` — a correlation of 0.7 removes ~51% of variance

**Common mistakes:**
- Using a covariate measured during the experiment as X — post-experiment covariates may be affected by the treatment, making the adjustment biased
- Applying CUPED when the pre-experiment covariate has low correlation with the outcome — minimal benefit, adds complexity for no gain
- Forgetting that CUPED changes the variance of Y but not the group means — the ATE estimate should be identical

---
## §4 — Multiple Testing

Every additional test you run increases the chance of a false positive. At α=0.05, running 20 tests on pure noise will produce ~1 false positive by chance.

In [ ]:
from statsmodels.stats.multitest import multipletests

# Simulate 20 metrics, all null (no true effect) — how many false positives?
np.random.seed(42)
n_tests = 20
p_values_null = np.array([stats.ttest_ind(
    np.random.normal(0,1,500),
    np.random.normal(0,1,500)
)[1] for _ in range(n_tests)])

print(f"False positives (uncorrected, α=0.05): {(p_values_null < 0.05).sum()} of {n_tests}")

# ── Correction 1: Bonferroni — most conservative ─────────────────────────────
# Divide alpha by number of tests: reject if p < alpha/n
# Strong family-wise error rate control — good for safety-critical tests
alpha_bonf = 0.05 / n_tests
reject_bonf = p_values_null < alpha_bonf
print(f"After Bonferroni (α={alpha_bonf:.4f}):         {reject_bonf.sum()} false positives")

# ── Correction 2: Benjamini-Hochberg FDR — preferred for DS experiments ──────
# Controls the False Discovery Rate (expected proportion of false positives among rejections)
# Less conservative than Bonferroni — better power when many tests are truly significant
reject_bh, p_adj_bh, _, _ = multipletests(p_values_null, alpha=0.05, method='fdr_bh')
print(f"After Benjamini-Hochberg FDR:              {reject_bh.sum()} false positives")

# ── When multiple testing applies ─────────────────────────────────────────────
# 1. Multiple primary metrics (shouldn't happen — pre-register one)
# 2. Multiple segments (by platform, country, user type)
# 3. Multiple variants (A/B/C/D test)
# 4. Sequential peeking (looking at results multiple times)

# ── Pre-registration rule ────────────────────────────────────────────────────
# Primary metric: pre-specified, no correction needed
# Secondary / exploratory metrics: apply BH-FDR correction
# Segment analysis: always treat as exploratory — correct and call out as hypothesis-generating
print()
print("Pre-registration tiers:")
print("  Primary metric (1):   no correction needed — this is THE decision metric")
print("  Guardrail metrics:    no correction — binary pass/fail, not hypothesis tests")
print("  Secondary metrics:    apply BH-FDR — exploratory, report as directional")
print("  Segment breakdowns:   apply BH-FDR — treat as hypothesis-generating only")

**Bonferroni vs BH-FDR:**

| | Bonferroni | Benjamini-Hochberg FDR |
| :--- | :--- | :--- |
| Controls | Family-wise error rate (FWER) | False discovery rate (FDR) |
| Conservatism | Very conservative | Less conservative |
| Power | Lower | Higher |
| Use when | Any false positive is costly | Some false positives acceptable |
| Common use | Clinical trials, safety tests | DS experiments, segment analysis |

**Common mistakes:**
- Testing many segments post-hoc and reporting the significant ones without correction — this is p-hacking, even if unintentional
- Applying Bonferroni to guardrail metrics — guardrails are not hypothesis tests; they are binary pass/fail checks
- Treating a significant segment result as confirmatory — segment analysis is always exploratory and needs a follow-up dedicated experiment

---
## §5 — Common Pitfalls & Diagnostics

These are the interview favorites — the scenarios where a naive analysis gives the wrong answer.

### Pitfall 1: Peeking

**What it is:** Checking results before the experiment ends and stopping early if significant.

**Why it's wrong:** Each peek is an additional test. With 5 peeks at α=0.05, the true false positive rate is ~19%, not 5%.

**Fix:** Pre-specify the stopping rule. If you must check early, use **sequential testing** (alpha spending functions — O'Brien-Fleming) or **always-valid p-values** (e-values, mSPRT).

---

### Pitfall 2: Novelty / Primacy Effect

**What it is:** Users behave differently simply because something is new (novelty effect) or because they're used to the old experience (primacy effect).

**Novelty:** Conversion spikes in week 1, decays to baseline by week 3 — feature looks better than it is.

**Primacy:** Users initially struggle with a new UI — feature looks worse than it is long-term.

**Detection:** Plot the metric by day or week within the experiment. Is the effect stable or trending?

**Fix:** Run the experiment for multiple weeks. Report the effect in weeks 2–4, not week 1.

---

### Pitfall 3: Network Effects / Spillover

**What it is:** Treated users affect control users — users share content, interact, or compete for the same resources.

**Effect:** ATE is underestimated (control benefits from treatment) or overestimated (treatment displaces control).

**Fix:** Cluster randomization (by geography, friend group, or time window).

---

### Pitfall 4: Simpson's Paradox in Segment Analysis

**What it is:** A trend appears in the overall data but reverses within each subgroup — because the subgroup composition differs between treatment and control.

In [ ]:
# Simpson's paradox example
# Overall: treatment looks worse. Within each platform: treatment is better.

data = pd.DataFrame({
    'group':    ['control']*4 + ['treatment']*4,
    'platform': ['iOS','iOS','Android','Android'] * 2,
    'users':    [8000, 2000, 1000, 9000,   # control: 80% iOS, 10% Android
                 1000, 9000, 8000, 2000],  # treatment: 10% iOS, 80% Android
    'converted':[1200,  220,   60,  450,
                  155,  990,  480,  100]
})
data['rate'] = data['converted'] / data['users']

# Overall rates
for grp in ['control', 'treatment']:
    d = data[data['group'] == grp]
    overall = d['converted'].sum() / d['users'].sum()
    print(f"{grp:12s} overall rate: {overall:.3%}")

print()

# Within-platform rates
for plat in ['iOS', 'Android']:
    for grp in ['control', 'treatment']:
        d = data[(data['group']==grp) & (data['platform']==plat)]
        r = d['converted'].sum() / d['users'].sum()
        print(f"{plat:8s} {grp:12s}: {r:.3%}")
    print()

# Conclusion: always check whether platform/country composition differs between groups
# before interpreting segment results

### Pitfall 5: Day-of-Week Bias

**What it is:** Launching on a Monday and stopping on a Thursday captures a biased day-of-week mix — weekdays only.

**Fix:** Always run for full weeks. If the experiment starts mid-week, exclude the partial first week from analysis.

---

### Pitfall 6: Survivorship Bias

**What it is:** Analyzing only users who completed an action (e.g., made a purchase) and ignoring those who dropped off.

**Example:** Comparing average order value between treatment and control, but only among users who ordered. The treatment may have attracted marginal buyers (lower AOV) — filtering them out makes the treatment look worse.

**Fix:** Always analyze on the full eligible population, not on the subset who completed the funnel.

---

### Quick Diagnostic Checklist

| Symptom | Likely cause | Investigation |
| :--- | :--- | :--- |
| Effect is huge in week 1, decays | Novelty effect | Plot metric by week |
| Effect looks worse despite better UX | Primacy effect | Run longer, check week 3+ |
| SRM detected | Logging bug, bot filtering | Pause experiment, audit pipeline |
| Segment results contradict overall | Simpson's paradox | Check group composition by segment |
| Control improves during treatment period | Spillover | Consider cluster randomization |
| Result significant at day 7, not at day 14 | Peeking / noise | Trust the pre-specified endpoint only |

---
## §6 — Communicating Results

The analysis is only useful if it drives a decision. Structure your communication around the decision, not the statistics.

### The Recommendation Structure

```
1. Bottom line up front     Ship / No-ship / Iterate — one sentence.

2. What moved               Primary metric: direction, magnitude, confidence interval.
                            "Conversion rate increased by 0.7pp (7% relative lift),
                             95% CI [0.3pp, 1.1pp], p=0.001."

3. Business impact          Translate to revenue / users / business terms.
                            "At current traffic, this is ~$2M incremental annual revenue."

4. Guardrails               Confirm no guardrail degraded.
                            "Latency, unsubscribe rate, and crash rate all within bounds."

5. Caveats                  Novelty effect? Segment heterogeneity? Power limitations?

6. Recommendation           Ship / no-ship / run a follow-up experiment.
```

### Framing for Common Scenarios

**Scenario 1: Result is not significant**
> *"We did not detect a statistically significant effect (p=0.31). This does not mean the feature has no effect — the experiment may be underpowered. The 95% CI is [-0.2pp, +0.6pp], which is consistent with both a small positive effect and no effect. Recommendation: if the feature is low-cost, consider running a longer experiment; if high-cost, do not ship."

**Scenario 2: Primary metric positive, guardrail degraded**
> *"Conversion rate increased by 0.7pp (significant), but page load time increased by 120ms, exceeding our 50ms guardrail threshold. We should not ship in its current form. The engineering team should optimize the implementation before re-testing."

**Scenario 3: Heterogeneous treatment effects**
> *"The overall ATE is +0.5pp. However, the effect is concentrated in iOS users (+1.2pp) and is near zero for Android users (+0.1pp, not significant). We may want to consider platform-specific rollout or investigate why the feature underperforms on Android."

### What NOT to Say

| Avoid | Say instead |
| :--- | :--- |
| "p=0.04, so the result is significant" | Report the effect size and CI alongside p-value |
| "The experiment proved the feature works" | "The experiment provides evidence that..." |
| "No effect was found" (when underpowered) | "We did not detect an effect; the experiment may be underpowered" |
| "The segment analysis shows..." (without correction) | "Exploratory segment analysis suggests... requires follow-up" |

**Common mistakes:**
- Leading with p-values instead of effect sizes — stakeholders care about business impact, not test statistics
- Saying "proven" or "confirmed" — experiments provide evidence, not proof
- Treating a non-significant result as "no effect" — always check whether the experiment was adequately powered

---
## Decision Guide

```
Running the analysis?
└── Always: SRM → balance → data quality → primary → guardrails → segments → recommend  (§1)

Choosing the statistical test?
├── Continuous metric (revenue, time)       → two-sample t-test                        (§2)
├── Proportion (conversion, CTR)            → z-test for proportions                  (§2)
├── Skewed or small sample                  → Mann-Whitney U                           (§2)
└── Always report: ATE + 95% CI + Cohen's d, not just p-value                         (§2)

Want faster results or smaller sample?
└── CUPED — use pre-experiment covariate, compute on train, apply to both groups       (§3)

Testing multiple things?
├── One pre-registered primary metric       → no correction needed                     (§4)
├── Multiple secondary metrics / segments  → Benjamini-Hochberg FDR                   (§4)
└── Safety / irreversible decisions        → Bonferroni                               (§4)

Results look wrong?
├── Effect huge then decays                → novelty effect → plot by week             (§5)
├── Segment contradicts overall            → Simpson's paradox → check composition    (§5)
├── SRM detected                          → stop, audit pipeline                      (§5)
├── Control group improves during test     → spillover → cluster randomization        (§5)
└── Significant early, not significant later → peeking artifact → trust endpoint     (§5)

Presenting results?
└── Bottom line → what moved → business impact → guardrails → caveats → recommend     (§6)
```